# Data Cleaning 

In [79]:
# Load the dataset
import pandas as pd

df = pd.read_csv(r"K:\End-to-End-Airbnb-Price-Prediction-ML-System\data\raw\Airbnb_Data.csv")
df.head()

,id,log_price,property_type,room_type,amenities,accommodates,bathrooms,bed_type,cancellation_policy,cleaning_fee,...,latitude,longitude,name,neighbourhood,number_of_reviews,review_scores_rating,thumbnail_url,zipcode,bedrooms,beds
0,6901257,5.010635,Apartment,Entire home/apt,"{""Wireless Internet"",""Air conditioning"",Kitche...",3,1.0,Real Bed,strict,True,...,40.696524,-73.991617,Beautiful brownstone 1-bedroom,Brooklyn Heights,2,100.0,https://a0.muscache.com/im/pictures/6d7cbbf7-c...,11201,1.0,1.0
1,6304928,5.129899,Apartment,Entire home/apt,"{""Wireless Internet"",""Air conditioning"",Kitche...",7,1.0,Real Bed,strict,True,...,40.766115,-73.989040,Superb 3BR Apt Located Near Times Square,Hell's Kitchen,6,93.0,https://a0.muscache.com/im/pictures/348a55fe-4...,10019,3.0,3.0
2,7919400,4.976734,Apartment,Entire home/apt,"{TV,""Cable TV"",""Wireless Internet"",""Air condit...",5,1.0,Real Bed,moderate,True,...,40.808110,-73.943756,The Garden Oasis,Harlem,10,92.0,https://a0.muscache.com/im/pictures/6fae5362-9...,10027,1.0,3.0
3,13418779,6.620073,House,Entire home/apt,"{TV,""Cable TV"",Internet,""Wireless Internet"",Ki...",4,1.0,Real Bed,flexible,True,...,37.772004,-122.431619,Beautiful Flat in the Heart of SF!,Lower Haight,0,NaN,https://a0.muscache.com/im/pictures/72208dad-9...,94117.0,2.0,2.0
4,3808709,4.744932,Apartment,Entire home/apt,"{TV,Internet,""Wireless Internet"",""Air conditio...",2,1.0,Real Bed,moderate,True,...,38.925627,-77.034596,Great studio in midtown DC,Columbia Heights,4,40.0,NaN,20009,0.0,1.0


In [80]:
df.columns

Index(['id', 'log_price', 'property_type', 'room_type', 'amenities',
       'accommodates', 'bathrooms', 'bed_type', 'cancellation_policy',
       'cleaning_fee', 'city', 'description', 'first_review',
       'host_has_profile_pic', 'host_identity_verified', 'host_response_rate',
       'host_since', 'instant_bookable', 'last_review', 'latitude',
       'longitude', 'name', 'neighbourhood', 'number_of_reviews',
       'review_scores_rating', 'thumbnail_url', 'zipcode', 'bedrooms', 'beds'],
      dtype='object')

In [81]:
# Drop unnecessary columns 
df.drop(columns=['id', 'name', 'thumbnail_url',
    'first_review',
    'last_review',
    'neighbourhood',
    'zipcode',
    'description'
], inplace=True)

In [82]:
# Check shape of df
df.shape

(74111, 21)

In [83]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74111 entries, 0 to 74110
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   log_price               74111 non-null  float64
 1   property_type           74111 non-null  object 
 2   room_type               74111 non-null  object 
 3   amenities               74111 non-null  object 
 4   accommodates            74111 non-null  int64  
 5   bathrooms               73911 non-null  float64
 6   bed_type                74111 non-null  object 
 7   cancellation_policy     74111 non-null  object 
 8   cleaning_fee            74111 non-null  bool   
 9   city                    74111 non-null  object 
 10  host_has_profile_pic    73923 non-null  object 
 11  host_identity_verified  73923 non-null  object 
 12  host_response_rate      55812 non-null  object 
 13  host_since              73923 non-null  object 
 14  instant_bookable        74111 non-null

TRANSFORM Columns

In [84]:
# convert amenities into numbers of amenities
df['amenities_count'] = df['amenities'].apply(lambda x: len(str(x).split(',')))

In [85]:
# Use host experience instead of host since
# Handle potential date parsing errors by coercing to NaT
df['host_years'] = 2025 - pd.to_datetime(df['host_since'], errors='coerce').dt.year

In [86]:
# Ensure host_response_rate is string, replace '%', then convert to numeric, coercing errors
df['host_response_rate'] = df['host_response_rate'].astype(str).str.replace('%', '')
df['host_response_rate'] = pd.to_numeric(df['host_response_rate'], errors='coerce')

In [87]:
# cleaning_fee is already boolean, direct conversion to int
df['cleaning_fee'] = df['cleaning_fee'].astype(int)

In [88]:
# instant_bookable needs mapping from 't'/'f' to 1/0
df['instant_bookable'] = df['instant_bookable'].map({'t': 1, 'f': 0}).astype(int)

FILL null values in the columns

In [89]:
for i in df.columns:
    null_count = df[i].isnull().sum()
    if null_count > 0:
        print(f"Column '{i}' has {null_count} null values. Type: {df[i].dtype}")

Column 'bathrooms' has 200 null values. Type: float64
Column 'host_has_profile_pic' has 188 null values. Type: object
Column 'host_identity_verified' has 188 null values. Type: object
Column 'host_response_rate' has 18299 null values. Type: float64
Column 'host_since' has 188 null values. Type: object
Column 'review_scores_rating' has 16722 null values. Type: float64
Column 'bedrooms' has 91 null values. Type: float64
Column 'beds' has 131 null values. Type: float64
Column 'host_years' has 188 null values. Type: float64


In [90]:
# Fill Bathrooms, Bedrooms and Beds by Median
for col in ['bathrooms', 'bedrooms', 'beds']:
    df[col] = df[col].fillna(df[col].median())

In [91]:
# Fill Review Scores Rating by Median
df['review_scores_rating'] = df['review_scores_rating'].fillna(
    df['review_scores_rating'].median()
)

In [92]:
# Fill Host Profile pic and Identity Verified by f stands for false
df['host_has_profile_pic'] = df['host_has_profile_pic'].fillna('f')
df['host_identity_verified'] = df['host_identity_verified'].fillna('f')

In [93]:
# host_has_profile_pic needs mapping from 't'/'f' to 1/0
df['host_has_profile_pic'] = df['host_has_profile_pic'].map({'t': 1, 'f': 0}).astype(int)

In [94]:
# host_identity_verified needs mapping from 't'/'f' to 1/0
df['host_identity_verified'] = df['host_identity_verified'].map({'t': 1, 'f': 0}).astype(int)

In [95]:
# Fill Host Response Rate by Median
df['host_response_rate'] = df['host_response_rate'].fillna(df['host_response_rate'].median())

In [96]:
# Fill Host Years by Median
df['host_years'] = df['host_years'].fillna(df['host_years'].median())

In [97]:
for i in df.columns:
    null_count = df[i].isnull().sum()
    if null_count > 0:
        print(f"Column '{i}' has {null_count} null values. Type: {df[i].dtype}")

Column 'host_since' has 188 null values. Type: object


In [98]:
df.drop(columns=['amenities','host_since'],inplace=True)

In [99]:
df[['host_response_rate','cleaning_fee','instant_bookable','host_has_profile_pic','host_identity_verified','host_years']].head()

,host_response_rate,cleaning_fee,instant_bookable,host_has_profile_pic,host_identity_verified,host_years
0,100.0,1,0,1,1,13.0
1,100.0,1,1,1,0,8.0
2,100.0,1,1,1,1,9.0
3,100.0,1,0,1,1,10.0
4,100.0,1,1,1,1,10.0


In [100]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74111 entries, 0 to 74110
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   log_price               74111 non-null  float64
 1   property_type           74111 non-null  object 
 2   room_type               74111 non-null  object 
 3   accommodates            74111 non-null  int64  
 4   bathrooms               74111 non-null  float64
 5   bed_type                74111 non-null  object 
 6   cancellation_policy     74111 non-null  object 
 7   cleaning_fee            74111 non-null  int64  
 8   city                    74111 non-null  object 
 9   host_has_profile_pic    74111 non-null  int64  
 10  host_identity_verified  74111 non-null  int64  
 11  host_response_rate      74111 non-null  float64
 12  instant_bookable        74111 non-null  int64  
 13  latitude                74111 non-null  float64
 14  longitude               74111 non-null

In [102]:
# Save the clean data
df.to_csv(r"K:\End-to-End-Airbnb-Price-Prediction-ML-System\data\raw\cleaned_airbnb_data.csv",index = False)